In [20]:
import logging
from pathlib import Path


import altair as alt
import polars as pl

from game.agents import AGENT_REGISTRY
from game.coordinate_methods import parse_coordinate
from game.fleet_placement_methods import PLACEMENT_METHODS
from game.game_board import GameBoard
from game.game_logger import GameLogger
from game.models import CellState

GameLogger.setup(console_level=logging.WARNING)
alt.data_transformers.enable("vegafusion")

REGENERATE_CSV = False
IMG_DIR = Path("img")
OUTPUT_CSV = Path("pre_analysis.csv")
GAMES_PER_CONFIG = 100
AGENT_TYPES = ["random", "hunt", "bayes"]
PLACEMENT_ORDER = list(PLACEMENT_METHODS.keys())

In [21]:
def run_game(agent_type: str, placement_method: str, game_id: int) -> dict:
    """Run one complete 2-player game and return per-game stats.

    Agent's fleet: random placement (fixed).
    Player's fleet: forced to placement_method.
    Player always uses RandomAgent shooting strategy.
    """
    agent = AGENT_REGISTRY[agent_type]()
    player = AGENT_REGISTRY["random"]()

    agent_board = GameBoard()  # player shoots at this
    player_board = GameBoard()  # agent shoots at this

    player_board.place_fleet(method=placement_method)
    agent.place_fleet(
        agent_board
    )  # delegates to board.place_fleet() with random method

    agent_hits = player_hits = turn = 0
    agent_won = False

    while True:
        turn += 1

        # --- Agent fires at player_board ---
        agent_obs = {
            "enemy_board": player_board.board_as_matrix(fog_of_war=True),
            "your_board": agent_board.board_as_matrix(fog_of_war=False),
            "ships_sunk": {
                "by_you": player_board.ships_sunk_count(),
                "against_you": agent_board.ships_sunk_count(),
            },
            "turn": turn,
        }
        coord = agent.select_move(agent_obs)
        r, c = parse_coordinate(coord)
        cell_state, ship = player_board.receive_shot(r, c)
        result = "HIT" if cell_state is CellState.HIT else "MISS"
        sunk = ship.ship_type.name if (ship and ship.is_sunk) else None
        if cell_state is CellState.HIT:
            agent_hits += 1
        agent.receive_result(coord, result, sunk)

        if player_board.all_ships_sunk():
            agent_won = True
            break

        # --- Player fires at agent_board ---
        player_obs = {
            "enemy_board": agent_board.board_as_matrix(fog_of_war=True),
            "your_board": player_board.board_as_matrix(fog_of_war=False),
            "ships_sunk": {
                "by_you": agent_board.ships_sunk_count(),
                "against_you": player_board.ships_sunk_count(),
            },
            "turn": turn,
        }
        coord = player.select_move(player_obs)
        r, c = parse_coordinate(coord)
        cell_state, ship = agent_board.receive_shot(r, c)
        result = "HIT" if cell_state is CellState.HIT else "MISS"
        sunk = ship.ship_type.name if (ship and ship.is_sunk) else None
        if cell_state is CellState.HIT:
            player_hits += 1
        player.receive_result(coord, result, sunk)

        if agent_board.all_ships_sunk():
            break

    return {
        "agent_type": agent_type,
        "placement_method": placement_method,
        "game_id": game_id,
        "agent_won": agent_won,
        "turns": turn,
        "agent_hits": agent_hits,
        "player_hits": player_hits,
        "agent_sunk": player_board.ships_sunk_count(),
        "player_sunk": agent_board.ships_sunk_count(),
    }

In [22]:
if REGENERATE_CSV:
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    records = []
    configs = [(a, m) for a in AGENT_TYPES for m in PLACEMENT_METHODS]
    n = len(configs)
    for i, (agent_type, method) in enumerate(configs):
        print(f"[{i + 1}/{n}] {agent_type} / {method} {'...':20s}", end="\r")
        for game_id in range(GAMES_PER_CONFIG):
            records.append(run_game(agent_type, method, game_id))
    df = pl.DataFrame(records)
    df.write_csv(OUTPUT_CSV)
    print(f"\nSaved {len(df):,} rows to {OUTPUT_CSV}")
else:
    df = pl.read_csv(OUTPUT_CSV)
    print(f"Loaded {len(df):,} rows from {OUTPUT_CSV}")

df.show()

Loaded 2,700 rows from pre_analysis.csv


agent_type,placement_method,game_id,agent_won,turns,agent_hits,player_hits,agent_sunk,player_sunk
str,str,i64,bool,i64,i64,i64,i64,i64
"""random""","""random""",0,false,86,14,17,2,5
"""random""","""random""",1,true,92,17,16,5,4
"""random""","""random""",2,false,89,16,17,4,5
"""random""","""random""",3,false,99,16,17,4,5
"""random""","""random""",4,true,93,17,16,5,4


In [23]:
summary = (
    df.group_by(["agent_type", "placement_method"])
    .agg(
        pl.col("agent_won").mean().alias("win_rate"),
        pl.col("turns").mean().alias("avg_turns"),
        pl.col("agent_hits").mean().alias("avg_agent_hits"),
        pl.col("player_hits").mean().alias("avg_player_hits"),
        pl.col("agent_sunk").mean().alias("avg_agent_sunk"),
        pl.col("player_sunk").mean().alias("avg_player_sunk"),
        pl.col("game_id").count().alias("n_games"),
    )
    .with_columns(
        (pl.col("win_rate") * 100).round(1).alias("win_rate_pct"),
        pl.col("avg_turns").round(1),
    )
    .sort(["agent_type", "placement_method"])
)
summary.show()

agent_type,placement_method,win_rate,avg_turns,avg_agent_hits,avg_player_hits,avg_agent_sunk,avg_player_sunk,n_games,win_rate_pct
str,str,f64,f64,f64,f64,f64,f64,u32,f64
"""bayes""","""clustered""",1.0,44.0,17.0,7.29,5.0,0.51,100,100.0
"""bayes""","""corners""",1.0,53.8,17.0,8.59,5.0,0.52,100,100.0
"""bayes""","""dense_center""",1.0,35.1,17.0,5.86,5.0,0.23,100,100.0
"""bayes""","""diagonal""",1.0,45.9,17.0,7.75,5.0,0.52,100,100.0
"""bayes""","""edges""",1.0,51.4,17.0,8.54,5.0,0.64,100,100.0


In [24]:
def chart_title(text: str) -> alt.TitleParams:
    return alt.TitleParams(text, fontSize=14, fontWeight="normal", anchor="middle")


# Heatmap: agent win rate by agent type x player placement method
heatmap = (
    alt.Chart(summary)
    .mark_rect()
    .encode(
        x=alt.X(
            "placement_method:N",
            sort=PLACEMENT_ORDER,
            title="Player Placement Method",
            axis=alt.Axis(labelAngle=-35),
        ),
        y=alt.Y("agent_type:N", sort=AGENT_TYPES, title="Agent Type"),
        color=alt.Color(
            "avg_turns:Q",
            title="Average Turns",
            scale=alt.Scale(scheme="blues", domain=[100, 0]),
        ),
        tooltip=[
            alt.Tooltip("agent_type:N", title="Agent"),
            alt.Tooltip("placement_method:N", title="Placement"),
            alt.Tooltip("win_rate_pct:Q", title="Win Rate (%)", format=".1f"),
            alt.Tooltip("avg_turns:Q", title="Avg Turns", format=".1f"),
        ],
    )
    .properties(
        title=chart_title("Agent Performance by Type and Player Placement Method"),
        width=500,
        height=150,
    )
)

labels = (
    alt.Chart(summary)
    .mark_text(fontSize=11)
    .encode(
        x=alt.X("placement_method:N", sort=PLACEMENT_ORDER),
        y=alt.Y("agent_type:N", sort=AGENT_TYPES),
        text=alt.Text("avg_turns:Q", format=".0f"),
        color=alt.condition(
            alt.datum.avg_turns < 60,
            alt.value("white"),
            alt.value("black"),
        ),
    )
)

c = heatmap + labels
c.save(IMG_DIR / "win_rate_heatmap.png")
c.show()

alt.LayerChart(...)

In [25]:
# Bar chart: avg turns by agent type (collapsed across placement methods)
turns_by_agent = df.group_by("agent_type").agg(
    pl.col("turns").mean().round(2).alias("avg_turns"),
    pl.col("turns").std().alias("std_turns"),
)

c = (
    alt.Chart(turns_by_agent)
    .mark_bar()
    .encode(
        x=alt.X("agent_type:N", sort=AGENT_TYPES[::-1], title="Agent Type"),
        y=alt.Y("avg_turns:Q", title="Average Turns"),
        color=alt.Color("agent_type:N", sort=AGENT_TYPES[::-1], legend=None),
        tooltip=[
            alt.Tooltip("agent_type:N", title="Agent"),
            alt.Tooltip("avg_turns:Q", title="Avg Turns", format=".2f"),
        ],
    )
    .properties(
        title=chart_title("Average Turns by Agent Type (all placements)"),
        width=300,
        height=250,
    )
)
c.save(IMG_DIR / "avg_turns_by_agent.png")
c.show()

alt.Chart(...)

In [26]:
# Grouped bars: avg agent hits vs player hits by agent type
hits_long = pl.concat(
    [
        df.select(
            [
                "agent_type",
                pl.col("agent_hits").alias("hits"),
                pl.lit("Agent").alias("side"),
            ]
        ),
        df.select(
            [
                "agent_type",
                pl.col("player_hits").alias("hits"),
                pl.lit("Player (random)").alias("side"),
            ]
        ),
    ]
)

hits_summary = hits_long.group_by(["agent_type", "side"]).agg(
    pl.col("hits").mean().alias("avg_hits")
)

c = (
    alt.Chart(hits_summary)
    .mark_bar()
    .encode(
        x=alt.X("agent_type:N", sort=AGENT_TYPES[::-1], title="Agent Type"),
        y=alt.Y("avg_hits:Q", title="Average Hits per Game"),
        xOffset=alt.XOffset("side:N"),
        color=alt.Color("side:N", title="Side", sort=AGENT_TYPES[::-1]),
        tooltip=[
            alt.Tooltip("agent_type:N", title="Agent"),
            alt.Tooltip("side:N", title="Side"),
            alt.Tooltip("avg_hits:Q", title="Avg Hits", format=".1f"),
        ],
    )
    .properties(
        title=chart_title("Average Hits per Game: Agent vs Random Player"),
        width=350,
        height=250,
    )
)
c.save(IMG_DIR / "avg_hits_by_agent.png")
c.show()

alt.Chart(...)